In [ ]:
import ast
import pickle
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('tmdb_5000_movies.csv')

In [ ]:
df = df[['title', 'genres', 'overview', 'vote_count', 'vote_average']]

In [ ]:
num_votes = df.groupby('title').count()['vote_count'].reset_index()
num_votes.rename(columns={'vote_count': 'num_vote'}, inplace=True)

In [ ]:
avg_vote = df.groupby('title').mean(numeric_only=True)['vote_count'].reset_index()
avg_vote.rename(columns={'vote_count': 'avg_vote'}, inplace=True)

In [ ]:
popular_df = num_votes.merge(avg_vote, on='title')[['title', 'num_vote', 'avg_vote']]

In [ ]:
popularity_df = popular_df.sort_values('avg_vote', ascending=False).head(50)

In [ ]:
def convert_genres(genre_str):
    try:
        genres_list = ast.literal_eval(genre_str)
        return [genre['name'] for genre in genres_list]
    except (ValueError, SyntaxError):
        return []

df['genres'] = df['genres'].apply(convert_genres)


In [ ]:
df['overview'] = df['overview'].fillna('')

df['tags'] = df['overview'] + ' ' + df['genres'].apply(lambda x: ' '.join(x))
df['tags'] = df['tags'].apply(lambda x: x.lower())


In [ ]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    return ' '.join([ps.stem(word) for word in text.split()])

df['tags'] = df['tags'].apply(stem)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags']).toarray()


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors).astype(np.float32)

In [ ]:
def recommend(movie_title):
    if movie_title not in df['title'].values:
        print(f"'{movie_title}' not found in dataset.")
        return []

    movie_index = df[df['title'] == movie_title].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]  
    recommendations = [df.iloc[i[0]].title for i in movie_list]

    return recommendations

In [ ]:
pickle.dump(df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
